# Validate the data pipeline

This notebook checks the prepared nflverse tables and demonstrates the player-week history contract used to build LSTM tensors. It does not make network calls.

In [13]:
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent
data_dir = project_root / 'data'
spec = spec_from_file_location('get_data', project_root / 'src' / 'get-data.py')
get_data = module_from_spec(spec)
spec.loader.exec_module(get_data)

print(f'Project root: {project_root}')
print(f'Available seasons: {get_data.available_seasons(2019)[-5:]}')

Project root: /Users/nicholaspatrick/Desktop/projects/Fantasy-Football-Predictor-2025
Available seasons: [2022, 2023, 2024, 2025, 2026]


In [14]:
table_names = ['weekly', 'team_weekly_stats', 'schedules', 'player_list', 'rosters']
tables = {name: pd.read_csv(data_dir / f'{name}.csv') for name in table_names}
assert set(tables) == set(table_names)

required_weekly_columns = {'fumbles', 'fumbles_lost', 'first_downs'}
assert required_weekly_columns <= set(tables['weekly'].columns)
print({name: frame.shape for name, frame in tables.items()})

{'weekly': (33287, 57), 'team_weekly_stats': (3070, 14), 'schedules': (1675, 47), 'player_list': (35720, 5), 'rosters': (229223, 38)}


In [15]:
player_list = tables['player_list']
if 'position' not in player_list.columns:
    player_list = get_data.build_player_list(tables['rosters'])
assert player_list['position'].isin(get_data.SKILL_POSITIONS).all()
assert player_list['week'].between(1, get_data.LAST_REGULAR_WEEK).all()
assert not player_list.duplicated(['season', 'week', 'player_name']).any()
print(player_list['position'].value_counts().sort_index())

QB     5223
RB     9141
TE     7954
WR    13400
Name: position, dtype: int64


In [17]:
weekly = tables['weekly']
candidates = (
    player_list[player_list['position'].isin({'WR', 'RB'})]
    .sort_values(['season', 'week', 'player_name'])
)
player_column = 'player_display_name' if 'player_display_name' in weekly.columns else 'player_name'
regular_weekly = weekly[weekly['week'].between(1, get_data.LAST_REGULAR_WEEK)].copy()
if 'season_type' in regular_weekly.columns:
    regular_weekly = regular_weekly[regular_weekly['season_type'] == 'REG']
example = None
for _, candidate in candidates.iterrows():
    prior = regular_weekly[
        (regular_weekly[player_column] == candidate['player_name'])
        & (
            (regular_weekly['season'] < candidate['season'])
            | ((regular_weekly['season'] == candidate['season']) & (regular_weekly['week'] < candidate['week']))
        )
    ]
    if not prior.empty:
        example = candidate
        break
assert example is not None
history = regular_weekly[
    (regular_weekly[player_column] == example['player_name'])
    & (regular_weekly['season'] <= example['season'])
].copy()
history = history[
    (history['season'] < example['season'])
    | (history['week'] < example['week'])
].sort_values(['season', 'week'])

assert not history.empty
assert history['week'].between(1, get_data.LAST_REGULAR_WEEK).all()
assert not ((history['season'] == example['season']) & (history['week'] >= example['week'])).any()
print(f"Example: {example['player_name']} ({example['position']}), forecast {example['season']} week {example['week']}")
display(history.tail(5)[[player_column, 'season', 'week', 'fantasy_points_ppr']])

Example: A.J. Brown (WR), forecast 2020 week 1


,player_display_name,season,week,fantasy_points_ppr
5157,A.J. Brown,2019,13,7.5
5158,A.J. Brown,2019,14,33.6
5159,A.J. Brown,2019,15,25.4
5160,A.J. Brown,2019,16,15.3
5161,A.J. Brown,2019,17,22.4


## Refreshing the data

Run `python src/get-data.py --start-season 2019` from the project root to fetch the latest available nflverse seasons and rewrite the five CSV tables. The command is intentionally separate from this notebook so validation stays deterministic.

In [22]:
from importlib.util import module_from_spec, spec_from_file_location
import yaml

create_spec = spec_from_file_location('create_tensors', project_root / 'src' / 'create_tensors.py')
create_tensors = module_from_spec(create_spec)
create_spec.loader.exec_module(create_tensors)

evaluate_spec = spec_from_file_location('evaluate_predictions', project_root / 'src' / 'evaluate_predictions.py')
evaluate_predictions = module_from_spec(evaluate_spec)
evaluate_spec.loader.exec_module(evaluate_predictions)

config = yaml.safe_load((project_root / 'config' / 'lstm_config_v1.yaml').read_text())
assert config['data']['lookback'] == 17
assert config['data']['look_forward'] == 17
assert config['data']['allow_cross_season_targets'] is True
print('Configured sequence: 17 weeks in -> 17 weeks out')

tensors = create_tensors.create_tensors(tables, config, player_name='CeeDee Lamb')
assert tensors['X_numeric'].shape[1] == config['data']['lookback']
assert tensors['y'].shape[1] == config['data']['look_forward']
assert tensors['X_categorical'].shape[:2] == tensors['X_numeric'].shape[:2]
assert tensors['input_played_mask'].shape == tensors['X_numeric'].shape[:2]
assert tensors['target_played_mask'].shape == tensors['y'].shape
assert any('rolling' in name for name in tensors['feature_names'])
assert any(name.startswith('opponent_defense__') for name in tensors['feature_names'])
print('CeeDee tensor shapes:', tensors['X_numeric'].shape, tensors['X_categorical'].shape, tensors['y'].shape)
print('Feature count:', len(tensors['feature_names']))

metadata = tensors['metadata'][0]
assert 'origin_season' in metadata
assert len(metadata['target_weeks']) == config['data']['look_forward']
print('First tensor metadata:', metadata)

Configured sequence: 17 weeks in -> 17 weeks out
CeeDee tensor shapes: (52, 17, 180) (52, 17, 3) (52, 17)
Feature count: 180
First tensor metadata: {'player_name': 'CeeDee Lamb', 'origin_season': 2020, 'origin_week': 17, 'target_seasons': [2021, 2021, 2021, 2021, 2021, 2021, 2021, 2021, 2021, 2021, 2021, 2021, 2021, 2021, 2021, 2021, 2021], 'target_weeks': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]}


## Team-switch alignment

A player's team is taken from weekly participation data when available, with roster data used for missing or future weeks.

In [19]:
timeline = create_tensors._build_player_timeline('Amari Cooper', 2024, tables, config)
played = tables['weekly'][(tables['weekly']['player_display_name'] == 'Amari Cooper') & (tables['weekly']['season'] == 2024)][['week', 'recent_team']]
team_check = timeline.merge(played, on='week')
assert (team_check['team'] == team_check['recent_team']).all()
assert team_check['team'].nunique() == 2
print(team_check[['week', 'team', 'recent_team', 'opponent_team']].to_string(index=False))
print('Team-switch alignment passed')

 week team recent_team opponent_team
    1  CLE         CLE           DAL
    2  CLE         CLE           JAX
    3  CLE         CLE           NYG
    4  CLE         CLE            LV
    5  CLE         CLE           WAS
    6  CLE         CLE           PHI
    7  BUF         BUF           TEN
    8  BUF         BUF           SEA
   11  BUF         BUF            KC
   13  BUF         BUF            SF
   14  BUF         BUF            LA
   16  BUF         BUF            NE
   17  BUF         BUF           NYJ
Team-switch alignment passed


## Raw and combined RMSE by horizon step

The evaluator scores only played targets and applies recency and horizon weights when overlapping forecasts are combined.

In [20]:
predictions = pd.DataFrame([
    {'player_name': 'x', 'target_season': 2025, 'target_week': 3, 'horizon_step': 1, 'y_true': 10.0, 'y_pred': 8.0, 'target_played': 1, 'forecast_age_days': 14},
    {'player_name': 'x', 'target_season': 2025, 'target_week': 3, 'horizon_step': 1, 'y_true': 10.0, 'y_pred': 9.5, 'target_played': 1, 'forecast_age_days': 2},
    {'player_name': 'x', 'target_season': 2025, 'target_week': 4, 'horizon_step': 2, 'y_true': 5.0, 'y_pred': 6.0, 'target_played': 1, 'forecast_age_days': 14},
])
evaluation = evaluate_predictions.evaluate_raw_and_combined(predictions)
assert set(evaluation) == {'raw', 'combined', 'combined_predictions'}
assert len(evaluation['raw']) == len(evaluation['combined']) == 2
assert evaluation['combined_predictions'].loc[0, 'n_forecasts'] == 2
print('Raw RMSE by horizon:')
display(evaluation['raw'])
print('Combined RMSE by horizon:')
display(evaluation['combined'])
print('Forecast aggregation validation passed')

Raw RMSE by horizon:


,horizon_step,n_predictions,rmse
0,1,2,1.457738
1,2,1,1.000000


Combined RMSE by horizon:


,horizon_step,n_predictions,rmse
0,1,1,1.16043
1,2,1,1.00000


Forecast aggregation validation passed


## Keras model contract

Build the configured model from the tensor metadata and verify the seq2seq output, optimizer, loss, and typed hyperparameter ranges.

In [23]:
import json
from importlib.util import module_from_spec, spec_from_file_location
import numpy as np
import tensorflow as tf

model_spec = spec_from_file_location('model', project_root / 'src' / 'model.py')
model_module = module_from_spec(model_spec)
model_spec.loader.exec_module(model_module)

cardinalities = {name: len(values) for name, values in tensors['vocabularies'].items()}
model = model_module.build_model(config, tensors['X_numeric'].shape[-1], cardinalities)
model_inputs = {
    'numeric': tensors['X_numeric'][:2],
    'categorical': tensors['X_categorical'][:2],
    'played_mask': tensors['input_played_mask'][:2],
}
predictions = model(model_inputs, training=False)
assert tuple(predictions.shape) == (2, config['data']['look_forward'])
optimizer = model_module.build_optimizer(model, config)
loss = model_module.build_loss(config)
search_combinations = model_module.expand_search_space(config)
assert optimizer is not None and loss is not None
assert len(search_combinations) == 5940
print('TensorFlow:', tf.__version__)
print('Model output shape:', tuple(predictions.shape))
print('Optimizer:', type(optimizer).__name__, 'Loss:', type(loss).__name__)
print('Search combinations:', len(search_combinations))
print('Keras model contract passed')

TensorFlow: 2.15.1
Model output shape: (2, 17)
Optimizer: AdamW Loss: Huber
Search combinations: 5940
Keras model contract passed
